# About dada filtering

Try to find how `dada2` filtering performs with bacteria data:

In [ ]:
require("here")
require("ggplot2")

In [ ]:
# Define a function to calculate the filtered metrics
calculate_filtered_metrics <- function(data) {
    data$filtered_drop <- data$DADA2_input - data$filtered
    data$filtered_percentage <- (data$filtered_drop / data$DADA2_input) * 100
    return(data)
}

Read data using a more stringent parameters on R2:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 220,
    "max_ee": 3,
}
```

In [ ]:
r2_constraints <- read.table(here("results-bacteria/dada2/DADA2_stats.tsv"), header = TRUE)
head(r2_constraints)

In [ ]:
r2_constraints <- calculate_filtered_metrics(r2_constraints)
summary(r2_constraints)

Read data calculated by setting read size to 230bp length:

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": 230,
    "trunclenr": 230,
    "max_ee": 2,
}
```

This was the analysis I want to call after determining reads length with `figaro`,
however, the parameters it suggested were less stringent and I didn'r recover any
sequences (maybe because I've removed all the sequences below 250bp?). So, I choose
to lower `trunclenf` and `trunclenr` parameters.

In [ ]:
fixed_length <- read.table(here("results-bacteria.1/dada2/DADA2_stats.tsv"), header = TRUE)
fixed_length <- calculate_filtered_metrics(fixed_length)
summary(fixed_length)

In this run, I've tried to increase the min quality used to determine where to
truncate forward and reverse reads:

```json
{
    "trunc_qmin": 30,
    "trunc_rmin": 0.75,
    "trunclenf": null,
    "trunclenr": null,
    "max_ee": 2,
}
```

In [ ]:
minq30 <- read.table(here("results-bacteria.2/dada2/DADA2_stats.tsv"), header = TRUE)
minq30 <- calculate_filtered_metrics(minq30)
summary(minq30)

Read data generated using default parameters (determine read length automatically):

```json
{
    "trunc_qmin": 25,
    "trunc_rmin": 0.75,
    "trunclenf": null,
    "trunclenr": null,
    "max_ee": 2,
}
```

In [ ]:
default_length <- read.table(here("results-bacteria.3/dada2/DADA2_stats.tsv"), header = TRUE)
default_length <- calculate_filtered_metrics(default_length)
summary(default_length)

In [ ]:
# Set the plot size
options(repr.plot.width = 12, repr.plot.height = 6)

# Combine the data into a single dataframe
combined_data <- rbind(
    data.frame(sample = default_length$sample, filtered_percentage = default_length$filtered_percentage, method = "Default"),
    data.frame(sample = minq30$sample, filtered_percentage = minq30$filtered_percentage, method = "minq30"),
    data.frame(sample = fixed_length$sample, filtered_percentage = fixed_length$filtered_percentage, method = "Fixed230"),
    data.frame(sample = r2_constraints$sample, filtered_percentage = r2_constraints$filtered_percentage, method = "R2_constraints")
)

# order data series
combined_data$method <- factor(combined_data$method, levels = c("Default", "minq30", "Fixed230", "R2_constraints"))

# Plot the barplot
ggplot(combined_data, aes(x = sample, y = filtered_percentage, fill = method)) +
    geom_bar(stat = "identity", position = "dodge") +
    labs(title = "Drop in Reads After Filtering by Method", x = "Sample", y = "Percentage of Reads Lost") +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

Incrementing only the *minimum quality* to 30 has no significant effect on the
number of reads discarded. Decrementing the *read length* to 230bp, however,
seems to save more reads. By fixing the R2 length to 220bp, and increasing the 
expected errors to 3 saves a lot of reads, even in samples which seems to have 
a low quality.